In [4]:
from dotenv import load_dotenv
import logging

from minsearch import Index

from ingest import load_course_data, build_index
from rag_helper import RAGBase


load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s %(message)s"
)

logging.getLogger('ingest').setLevel(logging.DEBUG)
logging.getLogger('rag_helper').setLevel(logging.DEBUG)

from openai import OpenAI
openai_client = OpenAI()


documents = load_course_data()

index = build_index(
    Index(
        text_fields=['content'],
        keyword_fields=['filename']
    ),
    documents
)

question = 'How does the agentic loop keep calling the model until it stops?'

class ContextBuilder:
    def build(self, search_result):
        lines = []

        for doc in search_result:
            lines.append('Course: ' + doc['content'])
            lines.append('Filename: ' + doc['filename'])
            lines.append('')

        return '\n'.join(lines).strip()

assistant = RAGBase(index, openai_client, ContextBuilder())
response = assistant.rag(question)
print(response)


2026-06-16 13:57:27,835 ingest DEBUG Index built with 295 documents
2026-06-16 13:57:29,619 httpx INFO HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-06-16 13:57:29,622 rag_helper DEBUG Number of input tokens consumed: 2318


('It keeps calling the model inside a `while True` loop.\n\nEach turn, the code checks whether the model returned any `function_call` items:\n\n- If yes, it runs the tool, appends the tool result to `messages`, and keeps looping.\n- If no function calls were returned, it breaks out of the loop.\n\nSo the stop condition is:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nIn other words, the loop ends when the model gives a final `message` and no more tool calls.', 2318)


In [1]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [9]:
from typing import Dict

instructions = """
You're a course teaching assistant. Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
""".strip()

documents = load_course_data()

index = build_index(
    Index(
        text_fields=['content'],
        keyword_fields=['filename']
    ),
    documents
)

def search(query: str) -> Dict[str, str]:
    """
    Search the course database for relevant information
    """
    return index.search(query)

agent_tools = Tools()
agent_tools.add_tool(search)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

result = runner.loop(
    prompt='How does the agentic loop work, and how is it different from plain RAG?',
    callback=callback
)

2026-06-16 14:01:41,608 ingest DEBUG Index built with 295 documents
2026-06-16 14:01:42,991 httpx INFO HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


-> Response received


2026-06-16 14:01:46,461 httpx INFO HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


-> Response received
